In [ ]:
#!/usr/bin/env python3

import argparse
import os
import sys
import glob

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import yaml

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 120,
})

COLOURS = {'radar': '#E07B39', 'lidar': '#4A90D9'}
LABELS  = {'radar': 'Radar (90° FOV)', 'lidar': 'LiDAR (360° FOV)'}


# ------------------------------------------------------------------ #
# Data loading                                                         #
# ------------------------------------------------------------------ #

def load_data(metrics_dir: str):
    ts_files = sorted(glob.glob(os.path.join(metrics_dir, '*_timestep.csv')))
    g_files  = sorted(glob.glob(os.path.join(metrics_dir, '*_goals.csv')))

    if not ts_files:
        sys.exit(f'No *_timestep.csv files found in {metrics_dir}')

    ts_df = pd.concat([pd.read_csv(f) for f in ts_files], ignore_index=True)
    g_df  = pd.concat([pd.read_csv(f) for f in g_files],  ignore_index=True) if g_files else pd.DataFrame()

    sensors = ts_df['sensor_type'].unique().tolist()
    print(f'Loaded {len(ts_files)} timestep file(s) | sensors: {sensors}')
    return ts_df, g_df, sensors


def load_map(map_yaml_path: str):
    """Returns (image_array, origin_x, origin_y, resolution) or None."""
    if not map_yaml_path or not os.path.exists(map_yaml_path):
        return None
    try:
        with open(map_yaml_path) as f:
            meta = yaml.safe_load(f)
        pgm_path = os.path.join(os.path.dirname(map_yaml_path), meta['image'])
        img = mpimg.imread(pgm_path)
        ox, oy = meta['origin'][0], meta['origin'][1]
        res = meta['resolution']
        return img, ox, oy, res
    except Exception as e:
        print(f'Warning: could not load map — {e}')
        return None


def world_to_pixel(wx, wy, img_height, ox, oy, res):
    px = (wx - ox) / res
    py = img_height - (wy - oy) / res
    return px, py


# ------------------------------------------------------------------ #
# Figure 1 — Trajectory overlay on map                                #
# ------------------------------------------------------------------ #

def fig_trajectory(ts_df, sensors, map_info, out_dir):
    fig, ax = plt.subplots(figsize=(8, 7))
    fig.suptitle('Figure 1 — Robot Trajectories', fontweight='bold')

    if map_info:
        img, ox, oy, res = map_info
        ax.imshow(img, cmap='gray', origin='upper',
                  extent=[ox, ox + img.shape[1] * res,
                          oy, oy + img.shape[0] * res])
        ax.set_xlim(ox, ox + img.shape[1] * res)
        ax.set_ylim(oy, oy + img.shape[0] * res)
    else:
        ax.set_aspect('equal')

    for s in sensors:
        c = COLOURS.get(s, 'gray')
        sub = ts_df[ts_df['sensor_type'] == s]
  
        ax.plot(sub['odom_x'], sub['odom_y'],
                color=c, linewidth=1.5, alpha=0.8, label=f'{LABELS.get(s, s)} (truth)')
        ax.plot(sub['kf_x'], sub['kf_y'],
                color=c, linewidth=1.0, linestyle='--', alpha=0.5,
                label=f'{LABELS.get(s, s)} (KF est.)')

    # Mark goals from first goal_pose sequence
    goals = ts_df.groupby(['sensor_type', 'goal_id'])[['odom_x', 'odom_y']].last().reset_index()
    for _, row in goals.iterrows():
        ax.scatter(row['odom_x'], row['odom_y'], marker='*', s=150,
                   color='gold', edgecolors='black', zorder=5)

    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    _save(fig, out_dir, 'fig1_trajectory.png')


# ------------------------------------------------------------------ #
# Figure 2 — Localisation error over time                             #
# ------------------------------------------------------------------ #

def fig_error_time(ts_df, sensors, out_dir):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=False)
    fig.suptitle('Figure 2 — Localisation Error Over Time', fontweight='bold')

    for s in sensors:
        sub = ts_df[ts_df['sensor_type'] == s].copy()
        # Normalise timestamp to start from 0 for each sensor block
        sub['t'] = sub['timestamp'] - sub['timestamp'].iloc[0]
        c = COLOURS.get(s, 'gray')
        lbl = LABELS.get(s, s)
        ax1.plot(sub['t'], sub['kf_error'],  color=c, linewidth=1.2, label=lbl)
        ax2.plot(sub['t'], sub['meas_error'], color=c, linewidth=1.2, label=lbl)

    ax1.set_ylabel('KF estimate error (m)')
    ax1.set_title('Kalman Filter estimate vs ground truth')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.set_ylabel('AMCL measurement error (m)')
    ax2.set_xlabel('Time (s)')
    ax2.set_title('Raw AMCL measurement vs ground truth')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    _save(fig, out_dir, 'fig2_error_time.png')


# ------------------------------------------------------------------ #
# Figure 3 — AMCL localisation confidence over time                  #
# ------------------------------------------------------------------ #

def fig_amcl_confidence(ts_df, sensors, out_dir):
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.suptitle('Figure 3 — AMCL Localisation Confidence (covariance trace)',
                 fontweight='bold')

    for s in sensors:
        sub = ts_df[ts_df['sensor_type'] == s].copy()
        sub['t'] = sub['timestamp'] - sub['timestamp'].iloc[0]
        c = COLOURS.get(s, 'gray')
        ax.plot(sub['t'], sub['amcl_cov_trace'],
                color=c, linewidth=1.2, label=LABELS.get(s, s))

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Cov trace (x_var + y_var)')
    ax.set_title('Lower = more confident localisation')
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    _save(fig, out_dir, 'fig3_amcl_confidence.png')


# ------------------------------------------------------------------ #
# Figure 4 — Per-goal bar chart                                       #
# ------------------------------------------------------------------ #

def fig_goal_bars(g_df, sensors, out_dir):
    if g_df.empty:
        print('No goals CSV found — skipping Figure 4.')
        return

    metrics = [
        ('time_to_goal', 'Time to goal (s)'),
        ('path_length',  'Path length (m)'),
        ('replan_count', 'Replanning events'),
    ]
    n_goals = g_df['goal_id'].nunique()
    goal_ids = sorted(g_df['goal_id'].unique())
    x = np.arange(len(goal_ids))
    width = 0.35

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle('Figure 4 — Per-Goal Performance Metrics', fontweight='bold')

    for ax, (col, ylabel) in zip(axes, metrics):
        for i, s in enumerate(sensors):
            sub = g_df[g_df['sensor_type'] == s]
            vals = [sub[sub['goal_id'] == gid][col].mean() for gid in goal_ids]
            offset = (i - len(sensors) / 2 + 0.5) * width
            ax.bar(x + offset, vals, width,
                   label=LABELS.get(s, s), color=COLOURS.get(s, 'gray'), alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels([f'G{g + 1}' for g in goal_ids])
        ax.set_xlabel('Goal')
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
        ax.legend(fontsize=9)
        ax.grid(True, axis='y', alpha=0.3)

    fig.tight_layout()
    _save(fig, out_dir, 'fig4_goal_bars.png')


# ------------------------------------------------------------------ #
# Figure 5 — Error distribution box plots                             #
# ------------------------------------------------------------------ #

def fig_error_boxplot(ts_df, sensors, out_dir):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle('Figure 5 — Error Distribution Across All Runs', fontweight='bold')

    kf_data   = [ts_df[ts_df['sensor_type'] == s]['kf_error'].dropna().values   for s in sensors]
    meas_data = [ts_df[ts_df['sensor_type'] == s]['meas_error'].dropna().values for s in sensors]
    labels    = [LABELS.get(s, s) for s in sensors]
    colours   = [COLOURS.get(s, 'gray') for s in sensors]

    def _box(ax, data, title, ylabel):
        bp = ax.boxplot(data, patch_artist=True, notch=False,
                        medianprops=dict(color='black', linewidth=2))
        for patch, c in zip(bp['boxes'], colours):
            patch.set_facecolor(c)
            patch.set_alpha(0.75)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, axis='y', alpha=0.3)

    _box(ax1, kf_data,   'KF Estimate Error',      'Error (m)')
    _box(ax2, meas_data, 'AMCL Measurement Error', 'Error (m)')

    fig.tight_layout()
    _save(fig, out_dir, 'fig5_error_boxplot.png')


# ------------------------------------------------------------------ #
# Helpers                                                              #
# ------------------------------------------------------------------ #

def _save(fig, out_dir, filename):
    path = os.path.join(out_dir, filename)
    fig.savefig(path, bbox_inches='tight')
    print(f'Saved → {path}')
    plt.close(fig)


def print_summary(ts_df, g_df, sensors):
    print('\n' + '=' * 55)
    print('SUMMARY')
    print('=' * 55)
    for s in sensors:
        sub_ts = ts_df[ts_df['sensor_type'] == s]
        print(f'\n  Sensor : {LABELS.get(s, s)}')
        print(f'  Samples: {len(sub_ts)}')
        print(f'  KF error   — mean: {sub_ts["kf_error"].mean():.3f} m  '
              f'std: {sub_ts["kf_error"].std():.3f} m')
        print(f'  Meas error — mean: {sub_ts["meas_error"].mean():.3f} m  '
              f'std: {sub_ts["meas_error"].std():.3f} m')
        if not g_df.empty:
            sub_g = g_df[g_df['sensor_type'] == s]
            sr = sub_g['success'].mean() * 100
            print(f'  Goals: {len(sub_g)}  success rate: {sr:.0f}%')
            print(f'  Avg time to goal : {sub_g["time_to_goal"].mean():.1f} s')
            print(f'  Avg path length  : {sub_g["path_length"].mean():.2f} m')
            print(f'  Avg replans/goal : {sub_g["replan_count"].mean():.1f}')
    print()


metrics_dir = r'\\wsl.localhost\Ubuntu-24.04\home\sisig\Dissertation-RobotAutomation\ros_jazzy_ws\src\my_bot_controller\ros_metrics'
map_yaml = r'\\wsl.localhost\Ubuntu-24.04\home\sisig\Dissertation-RobotAutomation\ros_jazzy_ws\src\my_bot_controller\maps\t3_house.yaml'
ts_df, g_df, sensors = load_data(metrics_dir)
map_info = load_map(map_yaml)

print_summary(ts_df, g_df, sensors)
fig_trajectory(ts_df, sensors, map_info, metrics_dir)
fig_error_time(ts_df, sensors, metrics_dir)
fig_amcl_confidence(ts_df, sensors, metrics_dir)
fig_goal_bars(g_df, sensors, metrics_dir)
fig_error_boxplot(ts_df, sensors, metrics_dir)


Loaded 2 timestep file(s) | sensors: ['lidar', 'radar']

SUMMARY

  Sensor : LiDAR (360° FOV)
  Samples: 3191
  KF error   — mean: 0.235 m  std: 0.131 m
  Meas error — mean: 0.236 m  std: 0.131 m
  Goals: 4  success rate: 50%
  Avg time to goal : 80.7 s
  Avg path length  : 5.19 m
  Avg replans/goal : 3.2

  Sensor : Radar (90° FOV)
  Samples: 3286
  KF error   — mean: 1.151 m  std: 0.824 m
  Meas error — mean: 1.151 m  std: 0.824 m
  Goals: 4  success rate: 75%
  Avg time to goal : 85.7 s
  Avg path length  : 5.86 m
  Avg replans/goal : 6.5

Saved → \\wsl.localhost\Ubuntu-24.04\home\sisig\Dissertation-RobotAutomation\ros_jazzy_ws\src\my_bot_controller\ros_metrics\fig1_trajectory.png
Saved → \\wsl.localhost\Ubuntu-24.04\home\sisig\Dissertation-RobotAutomation\ros_jazzy_ws\src\my_bot_controller\ros_metrics\fig2_error_time.png
Saved → \\wsl.localhost\Ubuntu-24.04\home\sisig\Dissertation-RobotAutomation\ros_jazzy_ws\src\my_bot_controller\ros_metrics\fig3_amcl_confidence.png
Saved → \\wsl.